# MercadoTico 360 · Requisitos del Caso 2 (CouchDB)

Ejecute las celdas **en orden**. Cada sección corresponde a un requisito de la investigación y termina con su evidencia en pantalla. Al final hay un resumen de cumplimiento.

> Requiere haber cargado antes los datos con `Simulacion.ipynb`.

| Requisito | Sección | Evidencia de la demostración |
|---|---|---|
| 1 y 2. Diseño y atributos por categoría | Documentos reales | Visualización de documentos y diseño |
| 3. Índices para 3+ patrones | Índices | Utilidad de cada índice |
| 4. Categoría + precio + atributo | Búsquedas | Pruebas **antes y después** de los índices |
| 5. Pedidos de un cliente y cambio de estado | Pedidos | Pedido anidado, estado actualizado |
| 6. Dos o más agregaciones | MapReduce | Ventas por categoría, ticket promedio, top productos |
| 7. Evolución del esquema | Evolución | Nueva categoría y atributo, validación |

In [ ]:
import json
import os
from getpass import getpass

import requests

try:                                   # Opcional: lee un archivo .env si existe
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass


def leer_config(nombre, pregunta, defecto=None, secreto=False):
    """Toma el valor de una variable de entorno; si no existe, lo pregunta."""
    valor = os.getenv(nombre)
    if valor:
        return valor.strip()
    if secreto:
        return getpass(f"{pregunta}: ").strip()
    sufijo = f" [{defecto}]" if defecto else ""
    return input(f"{pregunta}{sufijo}: ").strip() or defecto


COUCHDB_HOST  = leer_config("COUCHDB_HOST", "Host/IP de CouchDB", "localhost")
COUCHDB_PORT  = leer_config("COUCHDB_PORT", "Puerto de CouchDB")
COUCHDB_USER  = leer_config("COUCHDB_USER", "Usuario de CouchDB", "admin")
COUCHDB_PASS  = leer_config("COUCHDB_PASS", "Contraseña de CouchDB", secreto=True)
DATABASE_NAME = leer_config("DATABASE_NAME", "Nombre de la base de datos", "mercadotico360")

DB_URL = f"http://{COUCHDB_HOST}:{COUCHDB_PORT}/{DATABASE_NAME}"

SESSION = requests.Session()
SESSION.auth = (COUCHDB_USER, COUCHDB_PASS)


## Documentos y atributos variables
**Qué se embebe y qué se referencia**
- `producto`: documento propio; sus `atributos` van **embebidos** y cambian según la categoría.
- `pedido`: las `lineas_detalle` van **embebidas** (con *snapshot* de nombre, categoría y precio); `cliente_id` y `producto_id` son **referencias**.
- `cliente`: documento propio, para que sus datos cambien sin tocar los pedidos.

In [ ]:
def mostrar_productos_variables(categorias=("Ropa", "Tecnologia", "Artesanias")):
    # muestra un producto de cada categoría: los atributos son distintos en cada una.
    for cat in categorias:
        r = SESSION.post(f"{DB_URL}/_find", timeout=60,
                         json={"selector": {"type": "product", "categoria": cat}, "limit": 1}).json()
        if r.get("docs"):
            p = r["docs"][0]
            print(f"{p['_id']} | {p['categoria']} | atributos: {json.dumps(p['atributos'], ensure_ascii=False)}")


def mostrar_pedido_anidado(pedido_id="PED-000001"):
    # muestra un pedido completo con sus líneas de detalle anidadas.
    pedido = SESSION.get(f"{DB_URL}/{pedido_id}", timeout=10).json()
    print(json.dumps(pedido, indent=2, ensure_ascii=False))
    return pedido


print("--- Productos con atributos distintos por categoría ---")
mostrar_productos_variables()
print("\n--- Pedido con estructura anidada ---")
pedido_demo = mostrar_pedido_anidado()
registrar("Diseño y atributos variables", "lineas_detalle" in pedido_demo)

## Índices para patrones de consulta frecuentes

| Índice | Campos | Patrón de consulta que acelera |
|---|---|---|
| `idx_product_category` | `type`, `categoria` | Listar productos de una categoría (navegación del catálogo) |
| `idx_product_price` | `type`, `precio` | Buscar productos dentro de un rango de precios |
| `idx_technology_ram` | `type`, `categoria`, `atributos.ram_gb` | Filtrar por un atributo propio de una categoría (RAM en Tecnologia) |
| `idx_pedidos_cliente` | `type`, `cliente_id` | Recuperar el historial de pedidos de un cliente |

Mango selecciona un índice para ejecutar la consulta; las condiciones que no puedan resolverse directamente con ese índice se evalúan posteriormente sobre los documentos candidatos.

In [ ]:
INDICES = {
    "idx_product_category": ["type", "categoria"],
    "idx_product_price": ["type", "precio"],
    "idx_technology_ram": ["type", "categoria", "atributos.ram_gb"],
    "idx_pedidos_cliente": ["type", "cliente_id"],
}


def crear_indice(nombre):
    payload = {"index": {"fields": INDICES[nombre]}, "ddoc": nombre, "name": nombre, "type": "json"}
    respuesta = SESSION.post(f"{DB_URL}/_index", json=payload, timeout=300)
    respuesta.raise_for_status()
    res = respuesta.json()
    print(f"  {nombre}: {res.get('result')}")          # 'created' o 'exists'


def crear_indices():
    for nombre in INDICES:
        crear_indice(nombre)


def eliminar_indice(nombre):
    # elimina un índice por su nombre para comparar el rendimiento.
    for idx in SESSION.get(f"{DB_URL}/_index", timeout=30).json().get("indexes", []):
        if idx["name"] == nombre and idx["type"] == "json":
            SESSION.delete(f"{DB_URL}/_index/{idx['ddoc']}/json/{nombre}", timeout=60)
            print(f"  {nombre}: eliminado")


print("Creando índices (puede tardar unos segundos)...")
crear_indices()
registrar("Índices creados", True, f"{len(INDICES)} índices")

## Búsquedas por categoría, rango de precio y atributo específico
Se ejecuta la misma consulta antes y después de crear los índices para comparar los documentos examinados y el tiempo de ejecución.

In [ ]:
CONSULTA_CATEGORIA_PRECIO = {
    "type": "product",
    "categoria": "Tecnologia",
    "precio": {"$gte": 100000, "$lte": 500000},
}


def comparar_antes_despues():
    # mide la consulta categoría + precio sin índices y con índice.
    print("ANTES: se eliminan los índices de productos")
    for nombre in ("idx_product_category", "idx_product_price", "idx_technology_ram"):
        eliminar_indice(nombre)
    antes = buscar(CONSULTA_CATEGORIA_PRECIO, titulo="SIN índices")

    print("\nDESPUÉS: se vuelven a crear")
    crear_indices()
    despues = buscar(CONSULTA_CATEGORIA_PRECIO, use_index="idx_product_price", titulo="CON idx_product_price")

    ea, ed = antes.get("execution_stats", {}), despues.get("execution_stats", {})
    mejora = ea.get("total_docs_examined", 0) / max(ed.get("total_docs_examined", 1), 1)
    print(f"\nDocumentos examinados: {ea.get('total_docs_examined')} -> {ed.get('total_docs_examined')} "
          f"({mejora:.1f}x menos)")
    ok = len(antes.get("docs", [])) == len(despues.get("docs", [])) and mejora > 1
    registrar("Comparación antes/después de índices", ok, f"{mejora:.1f}x menos documentos examinados")


def buscar_categoria_precio_atributo(ram_gb=16, precio_min=100000, precio_max=1500000):
    # tecnologia + rango de precio + atributo específico (RAM).
    selector = {
    "type": "product",
    "categoria": "Tecnologia",
    "atributos.ram_gb": ram_gb,
    "precio": {"$gte": precio_min, "$lte": precio_max},
    }
    data = buscar(selector, use_index="idx_technology_ram", titulo=f"Tecnologia, {ram_gb} GB RAM, ₡{precio_min}-₡{precio_max}")
    if data.get("docs"):
        ejemplo = data["docs"][0]
        print(f"   ejemplo: {ejemplo['nombre']} | ₡{ejemplo['precio']} | {ejemplo['atributos']}")
    registrar("Búsqueda por categoría, precio y atributo", bool(data.get("docs")), f"{len(data.get('docs', []))} productos")


comparar_antes_despues()
print()
buscar_categoria_precio_atributo()

## Pedidos de un cliente y cambio de estado
Para actualizar solo el estado se usa un *update handler* de CouchDB: el cliente envía únicamente `{"estado": ...}` y el servidor aplica el cambio, sin que el código tenga que reenviar el pedido completo.
(CouchDB internamente guarda una nueva revisión del documento.)

In [ ]:
def instalar_update_handler():
    guardar_design_doc({
        "_id": "_design/pedidos",
        "updates": {
            "cambiar_estado": (
                "function (doc, req) {"
                "  if (!doc) { return [null, 'pedido no existe']; }"
                "  doc.estado = JSON.parse(req.body).estado;"
                "  return [doc, 'ok'];"
                "}"
            )
        },
    })


def cambiar_estado_pedido(pedido_id, nuevo_estado):
    url = f"{DB_URL}/_design/pedidos/_update/cambiar_estado/{pedido_id}"
    return SESSION.put(url, json={"estado": nuevo_estado}, timeout=30)


def recuperar_y_actualizar_pedido(cliente_id="CLI-000001", nuevo_estado="completado"):
    # 1. Recuperar los pedidos completos del cliente
    data = buscar(
    {"type": "order", "cliente_id": cliente_id},
    use_index="idx_pedidos_cliente",
    limit=1000,
    titulo=f"Pedidos de {cliente_id}")
    pedidos = data.get("docs", [])
    if not pedidos:
        registrar("Pedidos y actualización", False, "el cliente no tiene pedidos")
        return

    # 2. Actualizar el estado de uno (solo se envía el campo 'estado')
    pedido = next((p for p in pedidos if p["estado"] != nuevo_estado), pedidos[0])   # uno que sí cambie
    anterior = pedido["estado"]
    instalar_update_handler()
    res = cambiar_estado_pedido(pedido["_id"], nuevo_estado)

    # 3. Comprobar el cambio
    actualizado = SESSION.get(f"{DB_URL}/{pedido['_id']}", timeout=10).json()
    print(f"\n{pedido['_id']}: '{anterior}' -> '{actualizado['estado']}'  (HTTP {res.status_code})")
    lineas_iguales = pedido["lineas_detalle"] == actualizado["lineas_detalle"]
    print(
    f"Líneas de detalle conservadas: "
    f"{len(actualizado['lineas_detalle'])} | "
    f"sin cambios: {lineas_iguales}")   
    registrar("Pedidos y actualización de estado", actualizado["estado"] == nuevo_estado and lineas_iguales,
              f"{len(pedidos)} pedidos recuperados")


recuperar_y_actualizar_pedido("CLI-000001")